In [1]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_community.tools import DuckDuckGoSearchRun
import requests
import os

In [2]:
# Creating a search tool - Predefined
search_tool = DuckDuckGoSearchRun()

results = search_tool.invoke("Top IPL News") # Testing the tool

In [ ]:
# Creating a weather tool - Custom tool

@tool
def get_weather(city: str) -> str:
    """This function fetches the current weather data for the given city"""

    # Using WeatherStack API.
    # NOTE: Only 100 free requests per month. Use wisely..:)
    api_key=os.environ.get('WEATHER_STACK_API_KEY')
    url = f"https://api.weatherstack.com/current?access_key={api_key}&query={city}"

    response = requests.get(url)
    return response.json()


In [4]:
results

"Get the latest IPL 2026 news, updates, match previews, reviews, and more on News18 Cricket. Stay informed about your favorite teams, players, and matches. IPL 2026 News: Get all the live and latest news updates on IPL team, player squads, fixtures, IPL Result of Indian Premier League at Business Standard. Stay informed with all important IPL 2026 announcements, including player updates and squad changes. Get the latest news on IPLT20. Stay updated with today's cricket news, live scores, and expert analysis. Covering the IPL 2026, Test matches, and domestic leagues worldwide. Get breaking stories, player interviews, and match ... News Get the latest cricket news, match updates, player insights & breaking stories on IPL.com. Stay ahead with real-time cricket headlines, live scores, and in-depth analysis from top leagues worldwide."

In [5]:
# Defining LLM and testing it 
llm = ChatOpenAI()
llm.invoke("Hi")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 8, 'total_tokens': 17, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DXAwlTmqGkqo76hZyeDjrIcBCgWJZ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019db17d-55fc-7b80-9f16-82c2ff20e4d3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 9, 'total_tokens': 17, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [6]:
# Importing required libraries
# NOTE: Currently agent creation is dealt by lanngraph
# But we can also import the agents from langchain-classic (which is usually not recommended)
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_classic import hub

In [7]:

# Pull (pre-defined) prompt from langchain hub

prompt = hub.pull('hwchase17/react') # This is the most basic agent prompt


### Prompt in reAct agent

Based on paper "ReAct: Synergizing Reasoning and Acting in Language Models"

```
from langchain_core.prompts import PromptTemplate

template = '''Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}'''

prompt = PromptTemplate.from_template(template)
```

In [8]:
# Creating a reAct agent
# To use pre-defined agent in Langchian, we call a function
# Use function: create_react_agent()
# Parameters:
#   llm: Provide the LLM model to be user by the agent
#   tools: provide the list of tools (function) which you want agent to use
#   prompt: Either provide a pre-defined prompt or a manual SystemMessage prompt
agent = create_react_agent(
    llm=llm,
    tools=[search_tool,get_weather],
    prompt=prompt
)

#### Difference between Agent and AgentExecutor

Agent is different from agent executor.  

**Agent** just comes up with the tools which will help ouyt in performing a task.  

**AgentExecutor** on the other hand, actually executes the task proivded by the agent and paasses back the output to the agent.

In [9]:
# Creating a Agent Executor
# AgentExecutor class is used to create a Agent executor
# Parameters:
#   agent: Provide the name of agent created in previous step
#   tools: Pass the same list of tools (functions)
#   verbose: 'True' if you want see the thought porcess of Executor, else 'False'
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool,get_weather],
    verbose=True
)

In [12]:
# Invoking the Agent
# NOTE: Since is the older version of the agent from langchain-classic,
#       the agent will not work with newer models of OpenAI

query = "What is the weather of IT capital of India"

response = agent_executor({'input':query})
print(response)



> Entering new AgentExecutor chain...
I should use the get_weather function to find the current weather of the IT capital of India.
Action: get_weather
Action Input: Bangalore{'request': {'type': 'City', 'query': 'Bangalore, India', 'language': 'en', 'unit': 'm'}, 'location': {'name': 'Bangalore', 'country': 'India', 'region': 'Karnataka', 'lat': '12.983', 'lon': '77.583', 'timezone_id': 'Asia/Kolkata', 'localtime': '2026-04-22 01:31', 'localtime_epoch': 1776821460, 'utc_offset': '5.50'}, 'current': {'observation_time': '08:01 PM', 'temperature': 26, 'weather_code': 113, 'weather_icons': ['https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0008_clear_sky_night.png'], 'weather_descriptions': ['Clear '], 'astro': {'sunrise': '06:03 AM', 'sunset': '06:33 PM', 'moonrise': '10:23 AM', 'moonset': '11:50 PM', 'moon_phase': 'Waxing Crescent', 'moon_illumination': 27}, 'air_quality': {'co': '500.85', 'no2': '24.25', 'o3': '90', 'so2': '8.35', 'pm2_5': '19.35', 'pm10': '20.85'